# BigMart Sales Prediction


## Objective

The data scientists at BigMart have collected sales data for 1559 products across 10 stores in different cities. Also, certain attributes of each product and store have been defined. The aim is to build a predictive model and find out the sales of each product at a particular store (each row of data).

__So the idea is to find out the features (properties) of a product, and store which impacts the sales of a product.__






## Dataset Details

![](https://i.imgur.com/WlgNuFs.png)

# 1. Import Libraries

In this section, we import all the libraries required for data analysis, visualization, preprocessing, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# 2. Load Dataset

Load the BigMart sales dataset into a Pandas DataFrame for analysis.

In [ ]:
df = pd.read_csv('../data/sales_prediction.csv')

df.head()

# 3. Train-Test Split

In this section, we separate the input features and the target variable, then split the dataset into training and testing sets for model development.

In [ ]:
X = df.drop(columns=['Item_Outlet_Sales'])
y = df['Item_Outlet_Sales']

SEED=42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train.shape, X_test.shape

In [ ]:
X_train.head(3)

In [ ]:
y_train.head(3)

# 4. Exploratory Data Analysis (EDA)

In this section, we explore the dataset to understand its structure, identify missing values, examine numerical and categorical features, and visualize their distributions before preprocessing.

In [ ]:
X_train_original = X_train.copy()

## 4.1 Dataset Overview

X_train_original.info()

In [ ]:
X_train_original.info()

## 4.2 Missing Value Analysis

Identify missing values in each feature to determine which columns require preprocessing.

In [ ]:
X_train_original.isnull().sum()

## 4.3 Numerical Feature Analysis

Analyze the distribution and summary statistics of numerical features using descriptive statistics and visualizations.

In [ ]:
num_data=X_train_original.select_dtypes(exclude='object')
num_data.head()

In [ ]:
num_data.describe()

In [ ]:
num_data.isnull().sum()

In [ ]:
fig, ax=plt.subplots(1,2,figsize=(15,5))
sns.histplot(data=X_train_original, x= 'Item_Weight',ax=ax[0])
sns.boxplot(data= X_train_original,y='Item_Weight',ax=ax[1])

In [ ]:
def visualize_numeric_creation(dataframe, feature):
  fig, ax=plt.subplots(1,2,figsize=(15,5))
  sns.histplot(data=dataframe, x= feature,ax=ax[0],palette='Set3')
  sns.boxplot(data= dataframe,y=feature,ax=ax[1],palette='Set2')


In [ ]:
visualize_numeric_creation(X_train_original,'Item_Weight')

In [ ]:
visualize_numeric_creation(X_train_original, 'Item_Visibility')

In [ ]:
visualize_numeric_creation(X_train_original, 'Item_MRP')

In [ ]:
visualize_numeric_creation(X_train_original, 'Outlet_Establishment_Year')

In [ ]:
sns.countplot(data=X_train_original, x='Outlet_Establishment_Year')

## 4.4 Categorical Feature Analysis

Explore categorical variables by checking unique values, category frequencies, and distributions.

In [ ]:
Cat_features=X_train_original.select_dtypes(include='object')
Cat_features.head()

In [ ]:
Cat_features.describe()

In [ ]:
Cat_features.isnull().sum()

In [ ]:
Cat_features['Item_Identifier'].value_counts()

In [ ]:
Cat_features['Item_Fat_Content'].value_counts()

In [ ]:
Cat_features['Item_Type'].value_counts()

In [ ]:
Cat_features['Outlet_Identifier'].value_counts()

In [ ]:
Cat_features['Outlet_Size'].value_counts()

In [ ]:
Cat_features['Outlet_Location_Type'].value_counts()

In [ ]:
Cat_features['Outlet_Type'].value_counts()

# 5. Data Cleaning & Feature Engineering

In this section, we clean the dataset, handle missing values, standardize categorical values, and create new features required for model training.

In [ ]:
X_train_original['Item_Identifier'].apply(lambda x: x[0:2]).value_counts()

OR


In [ ]:
X_train_original['Item_Identifier'].str[:2].value_counts()

## 5.1 Create Item Type

Extract the product category from the Item_Identifier column and create a new feature called Item_Type.

## step 1 map item id's to item types

In [ ]:
#step 1 map item id's to item types
def create_item_type(data_frame):
  data_frame['Item_Type'] = data_frame['Item_Identifier'].apply(lambda x: x[0:2])
  data_frame['Item_Type'] = data_frame['Item_Type'].map({'FD':'Food','NC':'Non-Consumable','DR':'Drinks'})
  return data_frame

In [ ]:
X_train_original=create_item_type(X_train_original)
X_train_original.head()

In [ ]:
#Handle null values
X_train_original.isnull().sum()


In [ ]:
X_train_original[['Item_Identifier','Item_Weight']].drop_duplicates().sort_values(by='Item_Identifier')

In [ ]:
X_train_original[['Item_Type','Item_Weight']].drop_duplicates().sort_values(by='Item_Type')

In [ ]:
X_train_original[['Item_Type','Item_Weight']].sort_values(by='Item_Weight')

## 5.2 Handle Missing Values

Fill missing values using mapping and median imputation.

In [ ]:
ITEM_ID_WEIGHT_PIVOT=X_train_original.pivot_table(index='Item_Identifier',values='Item_Weight').reset_index()
ITEM_ID_WEIGHT_MAPPING=dict(zip(ITEM_ID_WEIGHT_PIVOT['Item_Identifier'],ITEM_ID_WEIGHT_PIVOT['Item_Weight']))
list(ITEM_ID_WEIGHT_MAPPING.items())[:5]

# Step 1: Create Item ID → Weight mapping

In [ ]:
# Step 1: Create Item ID → Weight mapping
ITEM_TYPE_WEIGHT_PIVOT=X_train_original.pivot_table(index='Item_Type',values='Item_Weight',aggfunc='median').reset_index()
ITEM_TYPE_WEIGHT_MAPPING=dict(zip(ITEM_TYPE_WEIGHT_PIVOT['Item_Type'],ITEM_TYPE_WEIGHT_PIVOT['Item_Weight']))
ITEM_TYPE_WEIGHT_MAPPING.items()

# Step 2: Fill missing weights using the mapping

In [ ]:
# Step 2: Fill missing weights using the mapping
def create_Fill_na_values(data_frame):
  data_frame['Item_Weight'].fillna(data_frame['Item_Identifier'].map(ITEM_ID_WEIGHT_MAPPING))
  data_frame['Item_Weight'].fillna(data_frame['Item_Type'].map(ITEM_TYPE_WEIGHT_MAPPING),)
  return data_frame

In [ ]:
create_Fill_na_values(X_train_original)

In [ ]:
X_train_original.isnull().sum()

In [ ]:
X_train_original[['Item_Type','Item_Weight']].sort_values(by='Item_Weight')

In [ ]:
X_train_original.isnull().sum()


In [ ]:
X_train_original.groupby(by=['Outlet_Type', 'Outlet_Size']).size()

In [ ]:
# Step 3: Fill remaining values using the mode
from scipy.stats import mode
Outlet_Type_Size_pivot = X_train_original.pivot_table(
    values='Outlet_Size',
    index='Outlet_Type',
    aggfunc=lambda x: x.mode()
).reset_index()

Outlet_Type_Size_Mapping = dict(
    zip(
        Outlet_Type_Size_pivot['Outlet_Type'],
        Outlet_Type_Size_pivot['Outlet_Size']
    )
)

Outlet_Type_Size_Mapping

In [ ]:
def create_fill_na_OutletSize(data_frame):
  data_frame['Outlet_Size'].fillna(data_frame['Outlet_Type'].map(Outlet_Type_Size_Mapping),inplace=True)
  return data_frame

In [ ]:
X_train_original=create_fill_na_OutletSize(X_train_original)

In [ ]:
X_train_original.isnull().sum()

## 5.3 Standardize Categorical Values

Standardize category names and handle non-consumable items.

In [ ]:
X_train_original.Item_Fat_Content.value_counts()

In [ ]:
def Standarise(data_frame):
  data_frame['Item_Fat_Content']=data_frame['Item_Fat_Content'].replace({'LF':'Low Fat','reg':'Regular','low fat':'Low Fat'})
  return data_frame

In [ ]:
X_train_original=Standarise(X_train_original)

In [ ]:
X_train_original.Item_Fat_Content.value_counts()

In [ ]:
X_train_original.groupby(by=['Item_Fat_Content','Item_Type']).size()

In [ ]:
def Replace_LowFat_NC(data_frame):
    data_frame.loc[data_frame['Item_Type'] == 'Non-Consumable', 'Item_Fat_Content'] = 'NA'
    return data_frame

In [ ]:
Replace_LowFat_NC(X_train_original)
X_train_original.groupby(by=['Item_Type','Item_Fat_Content']).size()

In [ ]:
Replace_LowFat_NC(X_train_original)
X_train_original.groupby(by=['Item_Type','Item_Fat_Content']).size()

In [ ]:
X_train_original.info()

In [ ]:
X_train_original.isnull().sum()

In [ ]:
# Step 1: Create Item_Identifier -> Item_Weight mapping
ITEM_ID_WEIGHT_PIVOT = (
    X_train_original
    .pivot_table(index='Item_Identifier', values='Item_Weight', aggfunc='first')
    .reset_index()
)

ITEM_ID_WEIGHT_MAPPING = dict(
    zip(ITEM_ID_WEIGHT_PIVOT['Item_Identifier'], ITEM_ID_WEIGHT_PIVOT['Item_Weight'])
)

# Step 2: Create Item_Type -> Median Item_Weight mapping
ITEM_TYPE_WEIGHT_PIVOT = (
    X_train_original
    .pivot_table(index='Item_Type', values='Item_Weight', aggfunc='median')
    .reset_index()
)

ITEM_TYPE_WEIGHT_MAPPING = dict(
    zip(ITEM_TYPE_WEIGHT_PIVOT['Item_Type'], ITEM_TYPE_WEIGHT_PIVOT['Item_Weight'])
)

# Step 3: Overall median Item_Weight
OVERALL_MEDIAN_WEIGHT = X_train_original['Item_Weight'].median()


# Step 4: Function to fill missing values
def create_fill_na_values(data_frame):
    # Fill using Item_Identifier mapping
    data_frame['Item_Weight'] = data_frame['Item_Weight'].fillna(
        data_frame['Item_Identifier'].map(ITEM_ID_WEIGHT_MAPPING)
    )

    # Fill remaining using Item_Type median
    data_frame['Item_Weight'] = data_frame['Item_Weight'].fillna(
        data_frame['Item_Type'].map(ITEM_TYPE_WEIGHT_MAPPING)
    )

    # Fill any remaining using overall median
    data_frame['Item_Weight'] = data_frame['Item_Weight'].fillna(OVERALL_MEDIAN_WEIGHT)

    return data_frame

In [ ]:
create_Fill_na_values(X_train_original)

In [ ]:
X_train_original.info()

In [ ]:
X_train_original.isnull().sum()

In [ ]:
# Step 1: Create Item_Identifier -> Item_Weight mapping
ITEM_ID_WEIGHT_PIVOT = (
    X_train_original
    .pivot_table(index='Item_Identifier', values='Item_Weight', aggfunc='first')
    .reset_index()
)

ITEM_ID_WEIGHT_MAPPING = dict(
    zip(ITEM_ID_WEIGHT_PIVOT['Item_Identifier'], ITEM_ID_WEIGHT_PIVOT['Item_Weight'])
)

# Step 2: Create Item_Type -> Median Item_Weight mapping
ITEM_TYPE_WEIGHT_PIVOT = (
    X_train_original
    .pivot_table(index='Item_Type', values='Item_Weight', aggfunc='median')
    .reset_index()
)

ITEM_TYPE_WEIGHT_MAPPING = dict(
    zip(ITEM_TYPE_WEIGHT_PIVOT['Item_Type'], ITEM_TYPE_WEIGHT_PIVOT['Item_Weight'])
)

# Step 3: Overall median Item_Weight
OVERALL_MEDIAN_WEIGHT = X_train_original['Item_Weight'].median()


# Step 4: Function to fill missing values
def create_fill_na_valuessss(data_frame):
    # Fill using Item_Identifier mapping
    data_frame['Item_Weight'] = data_frame['Item_Weight'].fillna(
        data_frame['Item_Identifier'].map(ITEM_ID_WEIGHT_MAPPING)
    )

    # Fill remaining using Item_Type median
    data_frame['Item_Weight'] = data_frame['Item_Weight'].fillna(
        data_frame['Item_Type'].map(ITEM_TYPE_WEIGHT_MAPPING)
    )

    # Fill any remaining using overall median
    data_frame['Item_Weight'] = data_frame['Item_Weight'].fillna(OVERALL_MEDIAN_WEIGHT)

    return data_frame

In [ ]:
create_fill_na_valuessss(X_train_original)

In [ ]:
X_train_original.isnull().sum()

## 5.4 Prepare Final Dataset

Combine all preprocessing steps into a single function.

In [ ]:
def prepare_data(data_frame):
  data_frame=create_item_type(data_frame)
  data_frame=create_fill_na_OutletSize(data_frame)
  data_frame=Standarise(data_frame)
  data_frame=create_Fill_na_values(data_frame)
  data_frame=Replace_LowFat_NC(data_frame)
  return data_frame

In [ ]:
X_train.isnull().sum()

In [ ]:
X_train=prepare_data(X_train_original)


In [ ]:
X_train.isnull().sum()

In [ ]:
X_test.isnull().sum()

In [ ]:
X_test_final=prepare_data(X_test)

In [ ]:
X_train.isnull().sum()

# 6. Feature Encoding

Convert categorical features into numerical values suitable for machine learning models.

In [ ]:
cat_feat=X_train.select_dtypes(include='object')
cat_feat.head()

## 6.1 Encode Categorical Features

Apply encoding techniques to convert categorical variables into numerical format.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore')
ohe.fit(cat_feat)

In [ ]:
ohe_feature_name = ohe.get_feature_names_out(input_features=cat_feat.columns)
ohe_feature_name

In [ ]:
num_feat=X_train.select_dtypes(exclude='object').reset_index(drop=True)
num_feat.head()

In [ ]:
cat_feats_train = X_train.select_dtypes(include='object')
cat_feats_train_ohe = pd.DataFrame(
    ohe.transform(cat_feats_train).toarray(),
    columns=ohe_feature_name
)
cat_feats_train_ohe.head()

In [ ]:
X_train_ohe = pd.concat([num_feat, cat_feats_train_ohe], axis=1)
X_train_ohe.head()

In [ ]:
final_columns = X_train_ohe.columns.values
final_columns

In [ ]:
num_feat_test = X_test.select_dtypes(exclude='object').reset_index(drop=True)
cat_feats_test = X_test.select_dtypes(include='object')
X_cat_test_ohe = pd.DataFrame(
    ohe.transform(cat_feats_test).toarray(),
    columns=ohe_feature_name
)
X_test_final=pd.concat([num_feat_test,X_cat_test_ohe],axis=1)
X_test_final = X_test_final[final_columns]
X_test_final.head()

# 7. Model Training

Train and compare multiple regression models on the preprocessed dataset.

In [ ]:
sns.histplot(data=y_train)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_validate
import numpy as np

In [ ]:
def train_and_eval_model(model, X_train, y_train, cv=5):
    cv_results = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring=('r2', 'neg_root_mean_squared_error')
    )
    print('Model:', model)

    r2_scores = cv_results['test_r2']
    print('R2 CV scores:', r2_scores)
    print('R2 CV scores mean / stdev:', np.mean(r2_scores), '/', np.std(r2_scores))

    rmse_scores = cv_results['test_neg_root_mean_squared_error']
    rmse_scores = [-1 * score for score in rmse_scores]
    print('RMSE CV scores:', rmse_scores)
    print('RMSE CV scores mean / stdev:', np.mean(rmse_scores), '/', np.std(rmse_scores))

## 7.1 Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(random_state=SEED)
train_and_eval_model(model=rf, X_train=X_train_ohe, y_train=y_train)

## 7.2 Gradient Boosting Regressor

In [ ]:
gb = GradientBoostingRegressor(random_state=SEED)
train_and_eval_model(model=gb, X_train=X_train_ohe , y_train=y_train)

## 7.3 Histogram Gradient Boosting Regressor

In [ ]:
hgb = HistGradientBoostingRegressor(random_state=SEED)
train_and_eval_model(model=hgb, X_train=X_train_ohe, y_train=y_train)

## 7.4 XGBoost Regressor

In [ ]:
xgr = xgb.XGBRegressor(objective='reg:squarederror',random_state=SEED)
train_and_eval_model(model=xgr, X_train=X_train_ohe, y_train=y_train)

## 7.5 LightGBM Regressor

In [ ]:
lgbr = LGBMRegressor(random_state=SEED)
train_and_eval_model(model=lgbr, X_train=X_train_ohe, y_train=y_train)

# 8. Model Evaluation

Evaluate and compare the performance of the trained models using regression metrics.

In [ ]:
X_train_copy = X_train.copy().drop(columns='Item_Identifier')

cat_cols = X_train_copy.select_dtypes(include=['object']).columns.tolist()
num_cols = cal_cols = X_train_copy.select_dtypes(exclude=['object']).columns.tolist()

cat_cols, num_cols

In [ ]:
X_train_copy[cat_cols] = X_train_copy[cat_cols].astype('category')
n_categorical_features = len(cat_cols)
n_numerical_features = len(num_cols)
X_train_copy = X_train_copy[cat_cols + num_cols]

X_train_copy.info()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector

categorical_mask = [True] * n_categorical_features + [False] * n_numerical_features

ordinal_encoder = make_column_transformer(
    (
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan),
        make_column_selector(dtype_include="category"),
    ),
    remainder="passthrough",
)

hgb = make_pipeline(
    ordinal_encoder,
    HistGradientBoostingRegressor(
        random_state=42, categorical_features=categorical_mask
    ),
)

train_and_eval_model(model=hgb, X_train=X_train_copy, y_train=y_train)

In [ ]:
lgbr = LGBMRegressor(random_state=SEED)

train_and_eval_model(model=lgbr, X_train=X_train_copy, y_train=y_train)

In [ ]:
cat_feats = X_train.select_dtypes(include=['object']).drop(columns=['Item_Identifier'])
ohe = OneHotEncoder(handle_unknown='ignore')
ohe.fit(cat_feats)
ohe_feature_names = ohe.get_feature_names_out(input_features=cat_feats.columns)

In [ ]:
num_feats_train = X_train.select_dtypes(exclude=['object']).reset_index(drop=True)
cat_feats_train = X_train.select_dtypes(include=['object']).drop(columns=['Item_Identifier'])
X_train_cat_ohe = pd.DataFrame(ohe.transform(cat_feats_train).toarray(), columns=ohe_feature_names)
X_train_final = pd.concat([num_feats_train, X_train_cat_ohe], axis=1)
X_train_final.head()

In [ ]:
gb = GradientBoostingRegressor(random_state=SEED)
train_and_eval_model(model=gb, X_train=X_train_final, y_train=y_train)

In [ ]:
hgb = HistGradientBoostingRegressor(random_state=SEED)
train_and_eval_model(model=hgb, X_train=X_train_final, y_train=y_train)

In [ ]:
xgr = xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED)
train_and_eval_model(model=xgr, X_train=X_train_final, y_train=y_train)

In [ ]:
lgbr = LGBMRegressor(random_state=SEED)
train_and_eval_model(model=lgbr, X_train=X_train_final, y_train=y_train)

In [ ]:
from sklearn.feature_extraction import FeatureHasher

hash_vector_size = 50
fh = FeatureHasher(n_features=hash_vector_size, input_type='string')
hashed_df = pd.DataFrame(
    fh.transform(X_train['Item_Identifier'].apply(lambda x: [x])).toarray(),
    columns=['H' + str(i) for i in range(hash_vector_size)]
)
hashed_df.head()

In [ ]:
cat_feats = X_train.select_dtypes(include=['object']).drop(columns=['Item_Identifier'])
ohe = OneHotEncoder(handle_unknown='ignore')
ohe.fit(cat_feats)
ohe_feature_names = ohe.get_feature_names_out(input_features=cat_feats.columns)

In [ ]:
num_feats_train = X_train.select_dtypes(exclude=['object']).reset_index(drop=True)
cat_feats_train = X_train.select_dtypes(include=['object']).drop(columns=['Item_Identifier'])
X_train_cat_ohe = pd.DataFrame(ohe.transform(cat_feats_train).toarray(), columns=ohe_feature_names)
X_train_final = pd.concat([num_feats_train, hashed_df, X_train_cat_ohe], axis=1)
X_train_final.head()

In [ ]:
X_train_final.shape

In [ ]:
hgbr=HistGradientBoostingRegressor(random_state=SEED)
train_and_eval_model(model=hgb, X_train=X_train_final, y_train=y_train)

In [ ]:
lgbm = LGBMRegressor(random_state=SEED)
train_and_eval_model(model=lgbm,  X_train=X_train_final, y_train=y_train)

In [ ]:
X_test.shape

# 9. Additional Experiments

Explore alternative encoding techniques and compare their impact on model performance.

In [ ]:
hashed_test_df = pd.DataFrame(
    fh.transform(X_test['Item_Identifier'].apply(lambda x: [x])).toarray(),
    columns=['H'+str(i) for i in range(hash_vector_size)]
)
num_feats_test = X_test.select_dtypes(exclude=['object']).reset_index(drop=True)
cat_feats_test = X_test.select_dtypes(include=['object']).drop(columns=['Item_Identifier'])
X_test_cat_ohe = pd.DataFrame(ohe.transform(cat_feats_test).toarray(), columns=ohe_feature_names)
X_test_final = pd.concat([num_feats_test, hashed_test_df, X_test_cat_ohe], axis=1)
X_test_final.head()

In [ ]:
X_test_final.shape

In [ ]:
gbr = GradientBoostingRegressor(random_state=SEED)
train_and_eval_model(model=gbr, X_train=X_train_final, y_train=y_train)

In [ ]:
gbr_model = GradientBoostingRegressor(random_state=SEED)
gbr_model.fit(X_train_final, y_train)

In [ ]:
xgr_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED)
xgr_model.fit(X_train_final, y_train)

In [ ]:
X_test = create_fill_na_values(X_test)
X_test_final.isna().sum().sum()  # should print 0

In [ ]:
X_test = create_fill_na_values(X_test)

hashed_test_df = pd.DataFrame(
    fh.transform(X_test['Item_Identifier'].apply(lambda x: [x])).toarray(),
    columns=['H'+str(i) for i in range(hash_vector_size)]
)
num_feats_test = X_test.select_dtypes(exclude=['object']).reset_index(drop=True)
cat_feats_test = X_test.select_dtypes(include=['object']).drop(columns=['Item_Identifier'])
X_test_cat_ohe = pd.DataFrame(ohe.transform(cat_feats_test).toarray(), columns=ohe_feature_names)
X_test_final = pd.concat([num_feats_test, hashed_test_df, X_test_cat_ohe], axis=1)

X_test_final.isna().sum().sum()  # should now print 0

# 10. Final Predictions

Generate predictions on the test dataset using the best-performing model.

In [ ]:
Y_pred = gbr_model.predict(X_test_final)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print('R2 Score:', r2_score(y_test, Y_pred))
print('RMSE Score:', np.sqrt(mean_squared_error(y_test, Y_pred)))

In [ ]:
xgr.fit(X_train_final, y_train)

fig, ax = plt.subplots(figsize=(20, 10))
xgb.plot_importance(xgr, ax=ax)
plt.show()

# 11. Conclusion

In this project:

- Performed exploratory data analysis.
- Cleaned and preprocessed the dataset.
- Engineered useful features.
- Encoded categorical variables.
- Trained and compared multiple regression models.
- Selected the best-performing model for BigMart sales prediction.

# 12. Future Improvements

- Perform hyperparameter tuning.
- Try additional feature engineering techniques.
- Compare more regression models.
- Deploy the best model as a web application.